# Getting started with Miyagi's Retrieval Augmented Generation (RAG) workflow using Azure AI Search and Semantic Kernel

This notebook walks through the RAG pattern that powers the Miyagi Recommendation service, one cell at a time.

## Before you run the cells

1. The [Polyglot Notebooks extension](https://marketplace.visualstudio.com/items?itemName=ms-dotnettools.dotnet-interactive-vscode) is installed in Visual Studio Code.
2. An Azure OpenAI (Microsoft Foundry) resource exists, with a chat completion deployment and a `text-embedding-3-small` deployment.
3. An Azure AI Search service exists.
4. The `.env` file in this folder has been filled in with the endpoints, keys and deployment names for those resources.
5. The kernel selector in the top right of this notebook is set to **.NET (C#)**.

> You need the [.NET 10 SDK](https://dotnet.microsoft.com/en-us/download) to run this notebook.

> Run the cells one at a time rather than using **Run All**. Several cells call Azure OpenAI, and running them all at once can exceed the deployment's tokens per minute limit.

## Overview

Retrieval Augmented Generation (RAG) grounds a language model on your own data instead of relying on what the model memorised during training. In this notebook you will:

1. Load configuration from `.env`.
2. Build a Semantic Kernel instance wired to your chat deployment.
3. Connect Azure AI Search as a vector store.
4. Chunk a book excerpt, embed the chunks, and persist them to the search index.
5. Query the index by meaning rather than by keyword.
6. Feed the retrieved passages into a prompt so the model's advice is grounded on them.

This is the same sequence the Recommendation service performs at runtime.

## Step 1 - Load settings from the .env file

This cell reads the `.env` file in this folder into a dictionary named `env`, which every later cell uses. If a value is missing here, the later cells will fail with an authentication or URI error.

In [ ]:
#r "nuget: dotenv.net, 3.2.1"

dotenv.net.DotEnv.Load();
var env = dotenv.net.DotEnv.Read();

// Confirm the settings were picked up, without printing the secrets themselves.
foreach (var key in new[] {
    "AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_CHAT_MODEL", "AZURE_OPENAI_EMBEDDING_MODEL",
    "AZURE_OPENAI_API_KEY", "AZURE_COGNITIVE_SEARCH_ENDPOINT", "AZURE_COGNITIVE_SEARCH_API_KEY" })
{
    var present = env.ContainsKey(key) && !string.IsNullOrWhiteSpace(env[key]) && !env[key].StartsWith("<");
    Console.WriteLine($"{(present ? "set    " : "MISSING")}  {key}");
}

## Step 2 - Install the Semantic Kernel packages

These are the dependencies used for the rest of the notebook. Restoring them the first time takes a moment.

Note the split of responsibilities:

- `Microsoft.SemanticKernel` gives you the kernel, prompt functions and plugins.
- `Microsoft.SemanticKernel.Connectors.AzureAISearch` provides the Azure AI Search vector store.
- `Microsoft.Extensions.AI.OpenAI` provides the embedding generator.

In [ ]:
#r "nuget: Microsoft.SemanticKernel, 1.78.0"
#r "nuget: Microsoft.SemanticKernel.Connectors.AzureAISearch, 1.74.0-preview"
#r "nuget: Microsoft.SemanticKernel.PromptTemplates.Handlebars, 1.78.0"
#r "nuget: Microsoft.SemanticKernel.Yaml, 1.78.0"
#r "nuget: Microsoft.Extensions.AI.OpenAI, 10.5.0"
#r "nuget: Azure.Identity, 1.17.0"
#r "nuget: System.IdentityModel.Tokens.Jwt, 8.14.0"
#r "nuget: Microsoft.Extensions.Logging, 10.0.0"
#r "nuget: Microsoft.Extensions.Logging.Console, 10.0.0"

Console.WriteLine("Packages restored.");

## Step 3 - Check your Azure sign-in

`DefaultAzureCredential` picks the first authentication method that works in the current environment, trying environment variables, managed identity, Visual Studio, Visual Studio Code and the Azure CLI in turn. Because you ran `az login` earlier, it will pick up your Azure CLI session here.

This notebook authenticates to Azure OpenAI and Azure AI Search with API keys, which keeps the setup simple. `DefaultAzureCredential` is the option you would use in production, and both client constructors below accept a credential in place of a key. This cell just confirms that a token can be acquired, so you can tell a sign-in problem apart from a configuration problem.

If it fails, run `az login` in the terminal and run the cell again.

In [ ]:
#pragma warning disable CS1701, CS1702

using Azure.Identity;
using Microsoft.Extensions.Logging;
using System.IdentityModel.Tokens.Jwt;
using System.Linq;

var credential = new DefaultAzureCredential();

try
{
    var tokenRequestContext = new Azure.Core.TokenRequestContext(new[] { "https://management.azure.com/.default" });
    var tokenResult = credential.GetToken(tokenRequestContext);
    var jwtToken = new JwtSecurityTokenHandler().ReadJwtToken(tokenResult.Token);

    var name = jwtToken.Claims.FirstOrDefault(c => c.Type == "name")?.Value;
    var uniqueName = jwtToken.Claims.FirstOrDefault(c => c.Type == "unique_name")?.Value;

    Console.WriteLine($"Signed in as: {name}");
    Console.WriteLine($"Account:      {uniqueName}");
}
catch (Exception ex)
{
    Console.WriteLine($"Could not acquire a token: {ex.Message}");
    Console.WriteLine("Run 'az login' in the terminal, then run this cell again.");
}

## Step 4 - Build the kernel

The kernel is the object that holds your AI services and plugins. Here it is given a single service, the Azure OpenAI chat completion deployment named in `AZURE_OPENAI_CHAT_MODEL`.

Note that no `MaxTokens`, and no sampling settings, are configured. The current generation of models rejects the older `max_tokens` parameter, so the kernel is left to use the model's own defaults.

In [ ]:
using Microsoft.SemanticKernel;

var builder = Kernel.CreateBuilder();

builder.Services.AddAzureOpenAIChatCompletion(
    deploymentName: env["AZURE_OPENAI_CHAT_MODEL"],
    endpoint: env["AZURE_OPENAI_ENDPOINT"],
    // credentials: credential, // Use this for token based authentication (recommended)
    apiKey: env["AZURE_OPENAI_API_KEY"] // Use this for API key based authentication (not recommended for production use)
);

var kernel = builder.Build();

Console.WriteLine($"Kernel ready, using chat deployment '{env["AZURE_OPENAI_CHAT_MODEL"]}'.");

## Step 5 - Connect Azure AI Search as a vector store

Two pieces are needed to store and retrieve embeddings.

**An embedding generator.** This turns text into a vector. It is used twice: once when indexing each chunk, and again at query time to embed the question.

**A vector collection.** This is the Azure AI Search index. The class below describes the index schema through attributes, so the index is created from your C# type rather than defined by hand in the portal:

- `[VectorStoreKey]` marks the document key.
- `[VectorStoreData]` marks a retrievable field.
- `[VectorStoreVector]` marks the vector field, with its dimension count, distance function and index type.

Notice that `Embedding` returns the chunk text rather than a float array. Because an embedding generator is attached to the collection, the connector embeds that text automatically on write and embeds your query automatically on read.

In [ ]:
#pragma warning disable SKEXP0001, SKEXP0010, SKEXP0020, MEVD9000, MEVD9001, OPENAI001, AOAI001, CS1701, CS1702

using Azure;
using Azure.AI.OpenAI;
using Azure.Search.Documents.Indexes;
using Microsoft.Extensions.AI;
using Microsoft.Extensions.VectorData;
using Microsoft.SemanticKernel.Connectors.AzureAISearch;

// The index schema. text-embedding-3-small returns 1536 dimensions.
public sealed class InvestmentThesisChunk
{
    [VectorStoreKey]
    public string Id { get; set; } = string.Empty;

    [VectorStoreData(IsFullTextIndexed = true)]
    public string Text { get; set; } = string.Empty;

    [VectorStoreData]
    public string Description { get; set; } = string.Empty;

    [VectorStoreVector(1536, DistanceFunction = DistanceFunction.CosineSimilarity, IndexKind = IndexKind.Hnsw)]
    public string Embedding => Text;
}

// Turns text into vectors, for both indexing and querying.
IEmbeddingGenerator<string, Embedding<float>> embeddingGenerator =
    new AzureOpenAIClient(
            new Uri(env["AZURE_OPENAI_ENDPOINT"]),
            new AzureKeyCredential(env["AZURE_OPENAI_API_KEY"]))
        .GetEmbeddingClient(env["AZURE_OPENAI_EMBEDDING_MODEL"])
        .AsIEmbeddingGenerator();

var searchIndexClient = new SearchIndexClient(
    new Uri(env["AZURE_COGNITIVE_SEARCH_ENDPOINT"]),
    new AzureKeyCredential(env["AZURE_COGNITIVE_SEARCH_API_KEY"]));

var memoryCollectionName = "miyagi-investment-thesis";

var collection = new AzureAISearchCollection<string, InvestmentThesisChunk>(
    searchIndexClient,
    memoryCollectionName,
    new AzureAISearchCollectionOptions { EmbeddingGenerator = embeddingGenerator });

await collection.EnsureCollectionExistsAsync();

Console.WriteLine($"Vector collection '{memoryCollectionName}' is ready.");

## Step 6 - Chunk the dataset and persist the embeddings

Embeddings are numeric representations of text that place similar meanings close together in vector space. That is what lets you retrieve a passage about "margin of safety" when the user asked about "investment philosophy", even though the two share no words.

Embeddings matter in a RAG workflow because they:

1. Put text into a common mathematical form a machine can compare.
2. Compress the text while keeping what distinguishes it.
3. Preserve relationships between related passages.
4. Are dense, so they are efficient to store and search.

Text has to be split before it is embedded. A single vector for a whole book would average away everything specific, and the passage would be too long to fit into a prompt. So this cell:

1. Reads the book excerpt from the Recommendation service's sample datasets.
2. Splits it into lines, then groups those lines into paragraph sized chunks.
3. Writes the chunks to Azure AI Search, which embeds each one on the way in.

The chunk count printed at the end is the number of documents you will see in the index in the Azure portal.

In [ ]:
#pragma warning disable SKEXP0050, SKEXP0055, CS1701, CS1702

using Microsoft.SemanticKernel.Text;
using System.IO;

var dataset = "intelligent-investor.txt";
var recommendationServicePath = "../../../../services/recommendation-service/dotnet";
const int MaxTokensPerParagraph = 300;
const int MaxTokensPerLine = 100;

// Load the text data from the local file
var datasetPath = Path.Combine(recommendationServicePath, "Resources", "sample-datasets", dataset);
var text = await File.ReadAllTextAsync(datasetPath);

// Chunk the text into lines, then group the lines into paragraphs
var lines = TextChunker.SplitPlainTextLines(text, MaxTokensPerLine);
var chunks = TextChunker.SplitPlainTextParagraphs(lines, MaxTokensPerParagraph);

Console.WriteLine($"Split {dataset} into {chunks.Count} chunks. Indexing...");

// Persist the chunks. The embedding generator attached to the collection vectorises each one.
var records = chunks.Select((chunk, i) => new InvestmentThesisChunk
{
    Id = $"{Path.GetFileNameWithoutExtension(dataset)}-{i}",
    Text = chunk,
    Description = $"Dataset: {dataset} Chunk: {i}"
});

await collection.UpsertAsync(records);

Console.WriteLine($"Saved {chunks.Count} chunks to collection '{memoryCollectionName}'.");

## Step 7 - Search and retrieve documents

`SearchAsync` embeds the query string and returns the closest chunks. The score is cosine similarity, so it runs from 0 to 1 and higher means closer.

Try changing the query to something the book does not discuss, and watch the scores drop. That difference is what tells you whether retrieval actually found supporting material or not.

In [ ]:
var query = "Ben Graham's investment philosophy";

await foreach (var result in collection.SearchAsync(query, top: 2))
{
    Console.WriteLine($"   Relevance: {result.Score:F4}");
    Console.WriteLine($"   {result.Record.Text}");
    Console.WriteLine();
}

## Step 8 - Load the grounded prompt

The prompt lives outside the code, as a YAML file at `services/recommendation-service/dotnet/Resources/Prompts/InvestmentAdvise.prompt.yaml`. Keeping it in its own file means the prompt can be reviewed and changed without recompiling the service.

Two details matter here:

- The template is Handlebars, so it can loop over the few shot examples supplied in Step 9.
- `AllowDangerouslySetContent` is switched on because those few shot examples are passed as chat messages rather than as a plain string, and the template engine will not encode those automatically.

In [ ]:
#pragma warning disable SKEXP0001, CS1701, CS1702

using System.IO;
using Microsoft.SemanticKernel.PromptTemplates.Handlebars;

var promptPath = Path.Combine(recommendationServicePath, "Resources", "Prompts", "InvestmentAdvise.prompt.yaml");

KernelFunction advisorFunction = kernel.CreateFunctionFromPromptYaml(
    await File.ReadAllTextAsync(promptPath),
    promptTemplateFactory: new HandlebarsPromptTemplateFactory { AllowDangerouslySetContent = true }
);

Console.WriteLine($"Loaded prompt function '{advisorFunction.Name}'.");

## Step 9 - Set the kernel arguments

This is where retrieval and generation meet. The cell:

1. Recalls the passages most relevant to the user's risk level, and serialises them into the `memories` argument.
2. Supplies a few shot example, so the model learns the exact JSON shape expected of it.
3. Collects the remaining prompt inputs: the user, their portfolio and the advisor voice to imitate.

The minimum relevance is deliberately not set high. Cosine scores for genuinely relevant prose typically sit in the 0.5 to 0.8 range, so a threshold of 0.8 or above silently discards good passages and leaves the prompt ungrounded.

In [ ]:
#pragma warning disable CS1701, CS1702

using System.Text.Json;
using Microsoft.SemanticKernel.ChatCompletion;

// Recall the passages that will ground the advice
var semanticQuery = "Investment advise for aggressive risk level";
var recalled = new List<string>();

await foreach (var result in collection.SearchAsync(semanticQuery, top: 2))
{
    if ((result.Score ?? 0d) < 0.5) continue;
    recalled.Add(result.Record.Text);
}

var memories = JsonSerializer.Serialize(recalled);
Console.WriteLine($"Recalled {recalled.Count} passage(s) to ground the prompt.");
Console.WriteLine();

// A few shot example, teaching the model the JSON shape to return
List<ChatHistory> fewShotExamples = [
    [
        new ChatMessageContent(AuthorRole.User, @"{""stocks"":[{""symbol"":""MSFT"",""allocation"":0.6},{""symbol"":""ACN"",""allocation"":0.4}]}"),
        new ChatMessageContent(AuthorRole.Assistant, @"{""portfolio"":[{""symbol"":""MSFT"",""gptRecommendation"":""Booyah! Hold on, steady growth! Diversify, though!""},{""symbol"":""ACN"",""gptRecommendation"":""Buy! Services will see a boom!""}]}")
    ]
];

var stocks = new[] {
    new { symbol = "MSFT", allocation = 0.3 },
    new { symbol = "ACN",  allocation = 0.1 },
    new { symbol = "JPM",  allocation = 0.3 },
    new { symbol = "PEP",  allocation = 0.3 }
};

var arguments = new KernelArguments
{
    ["userId"] = "50",
    ["stocks"] = JsonSerializer.Serialize(stocks),
    ["risk"] = "aggressive",
    ["fewShotExamples"] = fewShotExamples,
    ["voice"] = "Jim Cramer",
    ["memories"] = memories
};

Console.WriteLine(memories);

## Step 10 - Create a native function

Not everything a prompt needs comes from a vector store. Some of it has to be looked up or calculated. In Semantic Kernel those lookups are plain C# methods marked with `[KernelFunction]`, called native functions.

The prompt template refers to `{{UserProfilePlugin-GetUserAge userId}}` and `{{UserProfilePlugin-GetAnnualHouseholdIncome userId}}`. Those resolve to the two methods below, so the model is told the user's age and income without either value being hardcoded into the prompt. In a real deployment these methods would call a profile service.

In [ ]:
// Copyright (c) Microsoft. All rights reserved.

using System.ComponentModel;
using Microsoft.SemanticKernel;

/// <summary>
///     UserProfilePlugin shows a native function example that looks up user info given a userId.
/// </summary>
public class UserProfilePlugin
{
    /// <summary>Name of the context variable used for UserId.</summary>
    public const string UserId = "UserId";

    private const string DefaultUserId = "40";
    private const int DefaultAnnualHouseholdIncome = 150000;
    private const int Normalize = 81;

    /// <summary>Look up the user's age for a given userId.</summary>
    [KernelFunction, Description("Given a userId, get user age")]
    public string GetUserAge(
        [Description("Unique identifier of a user")]
        string userId)
    {
        userId = string.IsNullOrEmpty(userId) ? DefaultUserId : userId;

        int age;
        if (int.TryParse(userId, out var parsedUserId))
            age = parsedUserId > 100 ? parsedUserId % Normalize : parsedUserId;
        else
            age = int.Parse(DefaultUserId);

        // A real implementation would call a profile service here.
        return age.ToString();
    }

    /// <summary>Look up the user's annual household income for a given userId.</summary>
    [KernelFunction, Description("Given a userId, get user annual household income")]
    public string GetAnnualHouseholdIncome(
        [Description("Unique identifier of a user")]
        string userId)
    {
        userId = string.IsNullOrEmpty(userId) ? DefaultUserId : userId;

        var random = new Random();
        var randomMultiplier = random.Next(1000, 8000);

        // A real implementation would call a profile service here.
        var annualHouseholdIncome = int.TryParse(userId, out var parsedUserId)
            ? parsedUserId * randomMultiplier
            : DefaultAnnualHouseholdIncome;

        return annualHouseholdIncome.ToString();
    }
}

Registering the plugin makes both methods callable from the prompt template.

In [ ]:
kernel.Plugins.AddFromType<UserProfilePlugin>();

Console.WriteLine("Registered plugins: " + string.Join(", ", kernel.Plugins.Select(p => p.Name)));

## Step 11 - Invoke the LLM

The final cell runs the whole pipeline. The kernel renders the Handlebars template, which calls the native functions for age and income, injects the recalled passages, appends the few shot example, and sends the result to your chat deployment.

You should get back a JSON object with one recommendation per stock, written in the requested voice, and reflecting the investment thesis retrieved from the index. That last part is the payoff of RAG: the advice is anchored to your data, not just to the model's training.

If the response is not valid JSON, run the cell again. Models occasionally add prose around the JSON, which is why the Recommendation service retries parsing.

In [ ]:
var result = await kernel.InvokeAsync(advisorFunction, arguments);

var response = result.GetValue<string>();
Console.WriteLine(response);